# Suspensiones día por día

Cómo evoluciona cada merchant alrededor del inicio de su suspensión (activo → suspendido).

Genera:
- `docs/suspensiones.png`: transacciones diarias de cada merchant, de 30 días antes a 10 después del inicio.
- `docs/variables/<variable>.png`: lo mismo, solo para las variables que quedaron en el modelo.
- `docs/merchants/<merchant>.png`: cada merchant con 20 variables, de 60 días antes a 10 después.
- `docs/suspensiones_dia_a_dia.md`: tabla día por día, de 14 días antes a 5 después.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
MERCHANT_PATH = ROOT / "data" / "merchant_rating_featureset_sample.parquet"
DOCS = ROOT / "docs"
COLS = [
    "merchant_id", "observation_date", "is_suspended",
    "me_attempted_txn_sum_1d", "me_approved_txn_sum_1d", "me_tpv_approval_rate_7d",
]

## Datos diarios de los merchants alguna vez suspendidos

In [ ]:
status = pd.read_parquet(MERCHANT_PATH, columns=["merchant_id", "is_suspended"])
suspended_ids = status.loc[status["is_suspended"] == True, "merchant_id"].unique().tolist()  # noqa: E712

daily = pd.read_parquet(MERCHANT_PATH, columns=COLS, filters=[("merchant_id", "in", suspended_ids)])
daily["observation_date"] = pd.to_datetime(daily["observation_date"])
daily = daily.sort_values(["merchant_id", "observation_date"])

# Inicio = primer día suspendido cuyo día anterior estaba activo
onsets = []
for merchant, g in daily.groupby("merchant_id"):
    g = g.set_index("observation_date")
    state = g["is_suspended"]
    start = state.index[(state == True) & (state.shift(1) == False)]  # noqa: E712
    if len(start):
        onsets.append((start[0], merchant, g))
onsets.sort()
print(f"{len(suspended_ids)} merchants alguna vez suspendidos, {len(onsets)} con inicio observado")

## Gráfico

In [ ]:
fig, axes = plt.subplots(5, 4, figsize=(16, 15), sharex=True)
for ax, (onset, merchant, g) in zip(axes.flat, onsets):
    w = g.loc[onset - pd.Timedelta(days=30): onset + pd.Timedelta(days=10)]
    day = (w.index - onset).days
    attempts = w["me_attempted_txn_sum_1d"].clip(lower=0).fillna(0)
    approved = w["me_approved_txn_sum_1d"].clip(lower=0).fillna(0)
    ax.bar(day, attempts, color="#D85A30", width=0.9, label="intentos")
    ax.bar(day, np.minimum(approved, attempts), color="#378ADD", width=0.9, label="aprobadas")
    for d, s in zip(day, w["is_suspended"]):
        if s == True:  # noqa: E712
            ax.axvspan(d - 0.5, d + 0.5, color="#888780", alpha=0.3, lw=0)
        elif pd.isna(s):
            ax.axvspan(d - 0.5, d + 0.5, color="#EF9F27", alpha=0.15, lw=0)
    ax.axvline(0, color="black", lw=0.8, ls="--")
    ax.set_title(f"{merchant[:9]}…  inicio {onset.date()}", fontsize=9)
    ax.tick_params(labelsize=8)
for ax in axes.flat[len(onsets):]:
    ax.axis("off")
axes.flat[0].legend(fontsize=8, loc="upper left")
fig.text(0.5, 0.005, "días respecto del inicio de la suspensión (gris = suspendido, naranja claro = estado nulo)", ha="center")
fig.suptitle(f"Transacciones diarias de los {len(onsets)} merchants con inicio de suspensión observado", fontsize=13)
fig.tight_layout(rect=[0, 0.015, 1, 0.97])
fig.savefig(DOCS / "suspensiones.png", dpi=110)
plt.show()

## Cada variable alrededor de la suspensión

Un gráfico por variable que quedó en el modelo, con el mismo formato que `suspensiones.png`: un panel por merchant,
de 30 días antes a 10 después del inicio. Se guardan en `docs/variables/`.
En las tasas, el valor −1 (sin transacciones) se deja como hueco en la línea.

In [ ]:
full = pd.read_parquet(MERCHANT_PATH, filters=[("merchant_id", "in", [m for _, m, _ in onsets])])
full["observation_date"] = pd.to_datetime(full["observation_date"])
full = full.sort_values(["merchant_id", "observation_date"])
variables = [
    "me_txn_approval_rate_1d", "me_txn_approval_rate_7d", "me_txn_approval_rate_15d", "me_txn_approval_rate_30d",
    "me_tpv_approval_rate_1d", "me_tpv_approval_rate_7d", "me_tpv_approval_rate_15d", "me_tpv_approval_rate_30d",
    "me_lifetime_txn_approval_rate", "me_lifetime_tpv_approval_rate",
    "me_attempted_tpv_avg_1d", "me_attempted_tpv_avg_7d", "me_attempted_tpv_avg_30d", "me_lifetime_tpv_avg",
    "me_attempted_txn_sum_1d", "me_attempted_txn_sum_7d", "me_attempted_txn_sum_15d", "me_attempted_txn_sum_30d",
    "me_attempted_ecommerce_txn_sum_7d", "approved_tpv_90d", "me_lifetime_approved_txn",
]
by_merchant = {m: g.set_index("observation_date") for m, g in full.groupby("merchant_id")}

out_dir = DOCS / "variables"
out_dir.mkdir(exist_ok=True)
for var in variables:
    fig, axes = plt.subplots(5, 4, figsize=(16, 15), sharex=True)
    for ax, (onset, merchant, _) in zip(axes.flat, onsets):
        w = by_merchant[merchant].loc[onset - pd.Timedelta(days=30): onset + pd.Timedelta(days=10)]
        day = (w.index - onset).days
        for d, s in zip(day, w["is_suspended"]):
            if s == True:  # noqa: E712
                ax.axvspan(d - 0.5, d + 0.5, color="#888780", alpha=0.3, lw=0)
            elif pd.isna(s):
                ax.axvspan(d - 0.5, d + 0.5, color="#EF9F27", alpha=0.15, lw=0)
        values = w[var].where(w[var] > -0.5) if "rate" in var else w[var]  # -1 = sin transacciones
        ax.plot(day, values, color="#378ADD", marker=".", ms=3, lw=1)
        if "rate" in var:
            ax.set_ylim(-0.05, 1.1)
        ax.axvline(0, color="black", lw=0.8, ls="--")
        ax.set_title(f"{merchant[:9]}…  inicio {onset.date()}", fontsize=9)
        ax.tick_params(labelsize=8)
    for ax in axes.flat[len(onsets):]:
        ax.axis("off")
    fig.text(0.5, 0.005, "días respecto del inicio de la suspensión (gris = suspendido, naranja claro = estado nulo)", ha="center")
    fig.suptitle(f"{var} alrededor de cada suspensión", fontsize=13)
    fig.tight_layout(rect=[0, 0.015, 1, 0.97])
    fig.savefig(out_dir / f"{var}.png", dpi=90)
    plt.close(fig)
print(f"{len(variables)} gráficos en {out_dir}")


## Cada merchant con todas sus variables

Un gráfico por merchant: las variables que quedaron en el modelo, de 60 días antes a 10 después del inicio.
Sirve para leer por qué pudo ser suspendido. Se guardan en `docs/merchants/`.

In [ ]:
MERCHANT_VARS = [
    "me_attempted_txn_sum_1d", "me_approved_txn_sum_1d", "me_attempted_txn_sum_30d", "me_txn_approval_rate_7d",
    "me_tpv_approval_rate_30d", "me_attempted_tpv_avg_1d", "me_attempted_tpv_avg_30d", "me_attempted_ecommerce_txn_sum_7d",
    "me_lifetime_approved_txn", "me_lifetime_txn_approval_rate",
]
out_dir = DOCS / "merchants"
out_dir.mkdir(exist_ok=True)
for onset, merchant, _ in onsets:
    w = by_merchant[merchant].loc[onset - pd.Timedelta(days=60): onset + pd.Timedelta(days=10)]
    day = (w.index - onset).days
    fig, axes = plt.subplots(5, 4, figsize=(16, 14), sharex=True)
    for ax, var in zip(axes.flat, MERCHANT_VARS):
        for d, s in zip(day, w["is_suspended"]):
            if s == True:  # noqa: E712
                ax.axvspan(d - 0.5, d + 0.5, color="#888780", alpha=0.3, lw=0)
            elif pd.isna(s):
                ax.axvspan(d - 0.5, d + 0.5, color="#EF9F27", alpha=0.15, lw=0)
        values = w[var].where(w[var] > -0.5) if "rate" in var else w[var]  # -1 = sin transacciones
        ax.plot(day, values, color="#378ADD", marker=".", ms=3, lw=1)
        if "rate" in var:
            ax.set_ylim(-0.05, 1.1)
        ax.axvline(0, color="black", lw=0.8, ls="--")
        ax.set_title(var, fontsize=9)
        ax.tick_params(labelsize=8)
    for ax in axes.flat[len(MERCHANT_VARS):]:
        ax.axis("off")
    fig.text(0.5, 0.005, "días respecto del inicio de la suspensión (gris = suspendido, naranja claro = estado nulo)", ha="center")
    fig.suptitle(f"{merchant} — inicio {onset.date()}", fontsize=13)
    fig.tight_layout(rect=[0, 0.015, 1, 0.97])
    fig.savefig(out_dir / f"{merchant}.png", dpi=90)
    plt.close(fig)
print(f"{len(onsets)} gráficos en {out_dir}")


## Tabla día por día

In [ ]:
def fmt(value, decimals=1):
    return "" if pd.isna(value) else f"{value:.{decimals}f}"


lines = [
    "# Suspensiones día por día", "",
    f"Los {len(onsets)} merchants con inicio de suspensión observado (activo → suspendido), desde 14 días antes "
    "hasta 5 después del inicio. Día 0 = primer día suspendido.", "",
    "Columnas del día: intentos, aprobadas y rechazadas (intentos − aprobadas). Aprobación 7d: aprobación por TPV "
    "a 7 días tal como viene en el dataset (−1 = sin transacciones; puede tener ruido). Las fechas sin fila no aparecen.", "",
]
for onset, merchant, g in onsets:
    w = g.loc[onset - pd.Timedelta(days=14): onset + pd.Timedelta(days=5)]
    lines += [
        f"## {merchant} — inicio {onset.date()}", "",
        "| Fecha | Día | Estado | Intentos | Aprobadas | Rechazadas | Aprobación 7d |",
        "|---|---:|---|---:|---:|---:|---:|",
    ]
    for date, r in w.iterrows():
        s = r["is_suspended"]
        state = "**suspendido**" if s == True else ("activo" if s == False else "nulo")  # noqa: E712
        attempts = max(r["me_attempted_txn_sum_1d"], 0) if pd.notna(r["me_attempted_txn_sum_1d"]) else np.nan
        approved = max(r["me_approved_txn_sum_1d"], 0) if pd.notna(r["me_approved_txn_sum_1d"]) else np.nan
        rejected = max(attempts - approved, 0) if pd.notna(attempts) and pd.notna(approved) else np.nan
        lines.append(
            f"| {date.date()} | {(date - onset).days} | {state} | {fmt(attempts)} | {fmt(approved)} "
            f"| {fmt(rejected)} | {fmt(r['me_tpv_approval_rate_7d'], 2)} |"
        )
    lines.append("")
(DOCS / "suspensiones_dia_a_dia.md").write_text("\n".join(lines), encoding="utf-8")
print("guardado", DOCS / "suspensiones_dia_a_dia.md")